<div style="background-color: #ffffff; color: #000000; padding: 30px;">
<img src="../media/images/kisz_logo.png" width="192" height="69" align="right" style="margin-right: 50px; margin-bottom: 50px;">
<h1>Time Series Analysis and Forecasting</h1>
</div>

<div style="background-color: #f6a800; color: #ffffff; padding: 10px;">
<h2>Solutions</h2>
<h2>Notebook D03: Convolutional Networks for Time Series</h2>
</div>

Worked solutions to the 2 exercises in
[Notebook D03: Convolutional Networks for Time Series](../notebooks/D03_Convolutional_networks.ipynb).

**Try each exercise yourself first.** These notebooks are most useful as a check on your reasoning, and
least useful as something to read straight through. An exercise you attempted and got wrong teaches more
than a solution you agreed with.

Where an exercise asks a question rather than requesting code, the answer is written out under the code
that produces it. Several of them have answers that are more interesting than they look.

The setup cell below reproduces the state the exercises assume, so this notebook runs on its own.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="setup">Setup</h3>
</div>

The windows, the TCN and the training loop from the notebook. `CausalResidualBlock` takes one extra
argument here — `causal`, so that Exercise 2 can switch the padding — and `TCN` passes it through.

**This notebook trains seven TCNs and takes around half an hour on CPU.** `tcn_result` below caches each
configuration, so the full stack — which three of the cells want — is trained once rather than three
times. Every run uses seed 0 and the notebook's own settings, so the full stack reproduces its 260.6 MW
exactly and the comparisons are like for like.

In [ ]:
import sys
import importlib.util
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error

sys.path.append("../notebooks")
import nb_config

sns.set_theme(style="whitegrid")

TORCH_AVAILABLE = importlib.util.find_spec("torch") is not None

if TORCH_AVAILABLE:
    import torch
    from torch import nn
    from torch.nn import functional as F
    from torch.utils.data import DataLoader, TensorDataset

    torch.set_num_threads(1)
    print(f"PyTorch {torch.__version__}")
else:
    print("PyTorch is not installed. Run 'uv sync --group dl' to follow this notebook.")

ops = pd.read_parquet(nb_config.OPS_15M_PATH)

load = (
    ops[(ops["country"] == "AT") & (ops["measure"] == "actual_entsoe_transparency")]["value"]
    .tz_convert(None)
    .resample("h").mean()
    .dropna()
    .asfreq("h")
    .loc["2016-01-01":"2019-12-31"]
)

LOOKBACK, HORIZON = 168, 24
KERNEL_SIZE = 3
DILATIONS = (1, 2, 4, 8, 16, 32, 64)

values = load.values.astype(np.float32)
n_observations = len(values)
TEST_HOURS = VALIDATION_HOURS = 24 * 90
train_end = n_observations - TEST_HOURS - VALIDATION_HOURS

mean, std = values[:train_end].mean(), values[:train_end].std()
scaled = (values - mean) / std


def make_windows(scaled, start, stop):
    positions = range(start, stop)
    inputs = np.stack([scaled[t - LOOKBACK:t] for t in positions])
    targets = np.stack([scaled[t:t + HORIZON] for t in positions])
    return torch.tensor(inputs)[:, :, None], torch.tensor(targets)


def to_original_units(scaled_values):
    return np.asarray(scaled_values) * std + mean


def score(predictions, targets):
    return mean_absolute_error(
        to_original_units(targets).ravel(), to_original_units(predictions).ravel()
    )


class CausalResidualBlock(nn.Module):
    """Two dilated convolutions plus a skip connection.

    `causal` is the exercise's addition: left-padding only, as in the notebook,
    or split evenly on both sides, which is what a standard convolution does.
    """

    def __init__(self, in_channels, out_channels, kernel_size, dilation, causal=True):
        super().__init__()
        self.padding = (kernel_size - 1) * dilation
        self.causal = causal
        self.first = nn.Conv1d(in_channels, out_channels, kernel_size, dilation=dilation)
        self.second = nn.Conv1d(out_channels, out_channels, kernel_size, dilation=dilation)
        self.activation = nn.ReLU()
        self.project = (
            nn.Conv1d(in_channels, out_channels, kernel_size=1)
            if in_channels != out_channels else None
        )

    def pad(self, x):
        if self.causal:
            return F.pad(x, (self.padding, 0))
        left = self.padding // 2
        return F.pad(x, (left, self.padding - left))

    def forward(self, x):
        y = self.activation(self.first(self.pad(x)))
        y = self.activation(self.second(self.pad(y)))
        skip = x if self.project is None else self.project(x)
        return y + skip


class TCN(nn.Module):
    """The notebook's TCN, with the dilation schedule and padding exposed."""

    def __init__(self, channels=32, kernel_size=KERNEL_SIZE, dilations=DILATIONS,
                 horizon=HORIZON, causal=True):
        super().__init__()
        blocks, in_channels = [], 1
        for dilation in dilations:
            blocks.append(
                CausalResidualBlock(in_channels, channels, kernel_size, dilation, causal)
            )
            in_channels = channels
        self.blocks = nn.Sequential(*blocks)
        self.head = nn.Linear(channels, horizon)

    def forward(self, x):
        y = self.blocks(x.transpose(1, 2))
        return self.head(y[:, :, -1])


def train_model(build, epochs=6, batch_size=256, learning_rate=1e-3, seed=0):
    """The notebook's training loop, unchanged."""
    torch.manual_seed(seed)
    generator = torch.Generator().manual_seed(seed)

    model = build()
    optimiser = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss_function = nn.MSELoss()
    loader = DataLoader(
        TensorDataset(X_train, y_train),
        batch_size=batch_size, shuffle=True, generator=generator,
    )

    started = time.time()
    best = {"validation_mae": np.inf, "weights": None}

    for epoch in range(epochs):
        model.train()
        for batch_X, batch_y in loader:
            optimiser.zero_grad()
            loss_function(model(batch_X), batch_y).backward()
            optimiser.step()

        model.eval()
        with torch.no_grad():
            validation_mae = score(model(X_validation).numpy(), y_validation.numpy())

        if validation_mae < best["validation_mae"]:
            best = {"validation_mae": validation_mae,
                    "weights": {k: v.clone() for k, v in model.state_dict().items()}}

    model.load_state_dict(best["weights"])
    model.eval()
    with torch.no_grad():
        predictions = model(X_test).numpy()

    return {"model": model, "test_mae": score(predictions, y_test.numpy()),
            "parameters": sum(p.numel() for p in model.parameters()),
            "seconds": time.time() - started}


def receptive_field(dilations, kernel_size=KERNEL_SIZE, convs_per_block=2):
    """How many past steps one output can depend on."""
    reach = 1
    for dilation in dilations:
        for _ in range(convs_per_block):
            reach += (kernel_size - 1) * dilation
    return reach


_trained = {}


def tcn_result(dilations=DILATIONS, causal=True):
    """Train one TCN per distinct configuration, and remember it.

    Several cells below want the full stack, so without this the notebook would
    retrain the most expensive configuration three times over.
    """
    key = (tuple(dilations), causal)
    if key not in _trained:
        _trained[key] = train_model(lambda: TCN(dilations=dilations, causal=causal))
    return _trained[key]


if TORCH_AVAILABLE:
    X_train, y_train = make_windows(scaled, LOOKBACK, train_end - HORIZON)
    X_validation, y_validation = make_windows(
        scaled, train_end, train_end + VALIDATION_HOURS - HORIZON
    )
    X_test, y_test = make_windows(
        scaled, train_end + VALIDATION_HOURS, n_observations - HORIZON
    )

    naive_positions = range(train_end + VALIDATION_HOURS, n_observations - HORIZON)
    naive_forecast = np.stack([values[t - 24:t - 24 + HORIZON] for t in naive_positions])
    NAIVE_MAE = mean_absolute_error(
        to_original_units(y_test.numpy()).ravel(), naive_forecast.ravel()
    )

    print(f"train {tuple(X_train.shape)}   test {tuple(X_test.shape)}")
    print(f"Naive baseline: {NAIVE_MAE:.1f} MW")

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="exercise-1">Exercise 1</h3>
</div>

> The dilation table said seven blocks reach back 509 hours for a window only 168 long. Is all that reach used? Retrain the TCN with the dilation schedule truncated to two, three, four and five blocks, and plot test MAE against receptive field. Then separate reach from size: a schedule like `(1, 2, 4, 8, 1, 2, 4)` has the same depth and the same parameter count as the full stack but a much shorter reach. Which of the two is doing the work?

In [ ]:
SCHEDULES = [(1, 2), (1, 2, 4), (1, 2, 4, 8), (1, 2, 4, 8, 16), DILATIONS]

if TORCH_AVAILABLE:
    rows = []
    for schedule in SCHEDULES:
        outcome = tcn_result(dilations=schedule)
        rows.append({
            "blocks": len(schedule),
            "receptive field": receptive_field(schedule),
            "covers the window": receptive_field(schedule) >= LOOKBACK,
            "test MAE": outcome["test_mae"],
            "parameters": outcome["parameters"],
        })

    truncated = pd.DataFrame(rows).set_index("blocks")

truncated.round(1)

In [ ]:
if TORCH_AVAILABLE:
    fig, ax = plt.subplots(figsize=(11, 5))

    ax.plot(truncated["receptive field"], truncated["test MAE"], marker="o",
            markersize=8, linewidth=1.8, color="steelblue")
    for blocks, row in truncated.iterrows():
        ax.annotate(f"{blocks} blocks", (row["receptive field"], row["test MAE"]),
                    xytext=(8, 6), textcoords="offset points", fontsize=9)

    ax.axvline(LOOKBACK, color="crimson", linestyle="--", linewidth=1.2)
    ax.text(LOOKBACK * 1.05, truncated["test MAE"].max() * 0.95,
            "window length (168)", fontsize=9, color="crimson")
    ax.axhline(NAIVE_MAE, color="gray", linestyle=":", linewidth=1.2)
    ax.text(15, NAIVE_MAE + 6, "naive baseline", fontsize=9, color="gray")

    ax.set_xscale("log")
    ax.set_title("Accuracy against how far the network can see",
                 fontsize=13, fontweight="bold")
    ax.set_xlabel("Receptive field (hours, log scale)")
    ax.set_ylabel("Test MAE (MW)")
    ax.grid(linestyle="--", alpha=0.4)

    plt.tight_layout()
    plt.show()

**Every block earns its place, and the error falls monotonically: 456.9 MW at a reach of 13 hours down
to 260.6 at the full 509.** Two blocks barely beat the naive baseline; the full stack halves it.

The interesting part of the curve is between five blocks and seven, where the error drops from 310.0 to
260.6. Those two blocks add a 32- and a 64-hour dilation, and what they buy is the difference between a
reach of **125 hours and one of 509**.

125 is the number to look at. The window is 168 hours, so a five-block network reading the final position
can see back to hour 43 of the window and no further. Now ask what lives in the part it cannot see. To
forecast tomorrow's 08:00, the most informative single observation in the whole window is **08:00 exactly
one week ago** — hour 168 back, the same hour of the same weekday. A reach of 125 misses it, along with
the entire first two days of the window.

So the five-block model is not slightly short-sighted. It is cut off from the weekly cycle, which for
electricity load is the second-strongest pattern in the data after the daily one. That is what the last
50 MW of improvement is buying.

In [ ]:
# Dilation changes reach but not parameter count, so repeating a short schedule
# gives a network of identical depth and size that simply cannot see as far.
CONTROL = (1, 2, 4, 8, 1, 2, 4)

if TORCH_AVAILABLE:
    control_rows = []
    for label, schedule in [("full", DILATIONS), ("same size, short reach", CONTROL)]:
        outcome = tcn_result(dilations=schedule)
        control_rows.append({
            "schedule": str(schedule),
            "blocks": len(schedule),
            "receptive field": receptive_field(schedule),
            "parameters": outcome["parameters"],
            "test MAE": outcome["test_mae"],
        })

    control = pd.DataFrame(control_rows, index=["full", "same size, short reach"])

control.round(1)

**Reach is doing the work, and the control settles it: 260.6 against 319.6, from two networks with
exactly 41,336 parameters each and exactly seven blocks.**

This is why the second half of the exercise is worth the extra training run. The truncation experiment on
its own is confounded — each block removed takes 6,208 parameters with it, so the shorter networks are
both blinder *and* smaller, and the curve cannot tell you which one mattered. Holding depth and size
fixed while varying only the dilation schedule isolates it. A 59 MW penalty for seeing 89 hours instead of
509, at identical capacity, is as clean an answer as this kind of experiment gives.

Two things follow.

**Dilation is not a trick for saving parameters; it is the mechanism that makes the model work.** The
notebook introduces it as an efficiency argument — exponential reach for linear depth, seven blocks
instead of eighty-four. The control shows the stronger claim: given a fixed budget of seven blocks and
41,336 parameters, *how you space them* is worth 59 MW. Spend the budget on reach.

**And reach beyond the window is free but pointless.** The full schedule reaches 509 hours into a window
of 168, so roughly two-thirds of its nominal receptive field falls off the edge of the data. That costs
nothing and buys nothing. The design rule is to reach *at least* the window length and then stop: an
eighth block would add parameters and training time for no additional view. If you want the model to see
further, lengthen the window first, and add blocks to match.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="exercise-2">Exercise 2</h3>
</div>

> Section 4 calls causal padding the single most important detail in the architecture. Retrain the TCN with centred padding instead and see whether the score moves at all. Then work out why by differentiating: for an untrained network, compute the gradient of one output position with respect to every input position, and compare which inputs each variant actually touches. State precisely when the centred version would become a leak.

In [ ]:
if TORCH_AVAILABLE:
    padding_rows = []
    for label, causal in [("causal (left pad only)", True), ("centred (both sides)", False)]:
        outcome = tcn_result(causal=causal)
        padding_rows.append({"padding": label, "test MAE": outcome["test_mae"]})

    padding = pd.DataFrame(padding_rows).set_index("padding")

padding.round(1)

**The score does not improve. Centred padding scores 270.3 against causal padding's 260.6 — very
slightly *worse*.**

If centred convolutions were leaking the future, the opposite would happen: a model allowed to read ahead
would look brilliant on the test set, in the way the `target.shift(-1)` column did in Notebook
[C01](../notebooks/C01_Feature_engineering.ipynb). Nothing of the kind occurs here, and the notebook
predicts exactly this. The interesting question is why, and differentiating answers it precisely.

In [ ]:
def positions_touched(causal, probe, seed=0):
    """Which input positions an output position actually depends on.

    Differentiating the block stack's output at `probe` with respect to the whole
    input tells us the true receptive field: any input with a nonzero gradient is
    one this output can see. The network is untrained, because this is a property
    of the architecture rather than of the weights.
    """
    torch.manual_seed(seed)
    model = TCN(causal=causal)

    x = torch.zeros(1, LOOKBACK, 1, requires_grad=True)
    features = model.blocks(x.transpose(1, 2))
    features[0, :, probe].sum().backward()

    touched = (x.grad.abs().squeeze() > 0).nonzero().squeeze(-1)
    return int(touched.min()), int(touched.max())


if TORCH_AVAILABLE:
    rows = []
    for causal in (True, False):
        for probe in (LOOKBACK - 1, 100):
            first, last = positions_touched(causal, probe)
            rows.append({
                "padding": "causal" if causal else "centred",
                "output position": probe,
                "sees inputs": f"{first}..{last}",
                "reads ahead": last > probe,
            })

    reach = pd.DataFrame(rows).set_index(["padding", "output position"])

reach

There it is, in the one row that differs:

| padding | output position | sees inputs | reads ahead |
|---|---|---|---|
| causal | 167 (last) | 0..167 | no |
| causal | 100 | 0..100 | **no** |
| centred | 167 (last) | 0..167 | no |
| centred | 100 | 0..167 | **yes** |

**The centred network's interior positions read the future; its final position cannot, because in a window
of 168 there is nothing after position 167 to read.** And our head uses only the final position —
`self.head(y[:, :, -1])`. So the one output the model actually consumes is the one output where centred
and causal padding are equivalent, and that is the whole reason the score barely moved.

Notice also what the causal rows say about reach. Position 167 sees inputs 0..167 and position 100 sees
0..100: never anything later, and never more than the window allows. The nominal 509-hour receptive field
from Exercise 1 is clipped by the data, which is the same point from the other direction.

**So when would the centred version be a leak?** Precisely and in three cases:

1. **If the head read any position but the last.** Pooling over positions, averaging them, taking an
   intermediate one — all common, all standard in convolutional architectures. Every one of them would
   pull position 100's forward-looking view into the forecast. A `MaxPool` over the time axis is enough.
2. **If the convolution were applied to a sequence running up to the present rather than to a closed
   historical window.** Here the window ends 168 hours in the past relative to nothing — it is a block of
   settled history, and its interior "future" is still the past. Slide the same layer along a live series
   and the positions near the right-hand edge need values that have not happened.
3. **If the output were the sequence rather than a summary** — a denoised or imputed series, say, as in
   Notebook [A03](../notebooks/A03_Handling_missing_data.ipynb). Then every position is an output, and
   every interior position is contaminated.

Which reframes the notebook's claim usefully. Causal padding is not what makes *this* model correct — the
head's `[:, :, -1]` is doing that, silently and by accident. Causal padding is what keeps the model
correct when somebody later changes the head, lengthens the window to the present, or reuses the block in
a sequence-to-sequence setting. **It costs one line and removes a class of bug that does not announce
itself**, which is the same argument as building lag features with `shift` rather than a centred window.

The 10 MW the centred version loses, incidentally, is not a mystery either: splitting the padding between
both ends means the left edge of the window gets half as much context, so the early positions are built
from more zero-padding than real data. Causal padding puts all of it where the history is.

---

Back to [Notebook D03](../notebooks/D03_Convolutional_networks.ipynb), or on to
[Notebook D04](../notebooks/D04_Transformers.ipynb).